In [ ]:
import pyspark
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
        .master("local[*]") \
        .appName('test') \
        .getOrCreate()

In [ ]:
print(spark.version)


In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

In [ ]:
!wc -l yellow_tripdata_2024-10.parquet

In [ ]:
df = spark.read \
    .option("header", "true") \
    .csv('yellow_tripdata_2024-10.parquet')

In [ ]:
df.schema

In [ ]:
import pandas as pd
df_pandas = pd.read_parquet("yellow_tripdata_2024-10.parquet")

In [ ]:
df_head = df_pandas.head(1000)

In [ ]:
df_head.dtypes

In [ ]:
spark.createDataFrame(df_head).schema


In [ ]:
from pyspark.sql import types

In [ ]:

schema = types.StructType([
    types.StructField('VendorID', types.LongType(), True), 
    types.StructField('tpep_pickup_datetime', types.TimestampType(), True), 
    types.StructField('tpep_dropoff_datetime', types.TimestampType(), True), 
    types.StructField('passenger_count', types.LongType(), True), 
    types.StructField('trip_distance', types.DoubleType(), True), 
    types.StructField('RatecodeID', types.LongType(), True), 
    types.StructField('store_and_fwd_flag', types.StringType(), True), 
    types.StructField('PULocationID', types.LongType(), True), 
    types.StructField('DOLocationID', types.LongType(), True), 
    types.StructField('payment_type', types.LongType(), True), 
    types.StructField('fare_amount', types.DoubleType(), True), 
    types.StructField('extra', types.DoubleType(), True), 
    types.StructField('mta_tax', types.DoubleType(), True), 
    types.StructField('tip_amount', types.DoubleType(), True), 
    types.StructField('tolls_amount', types.DoubleType(), True), 
    types.StructField('improvement_surcharge', types.DoubleType(), True), 
    types.StructField('total_amount', types.DoubleType(), True), 
    types.StructField('congestion_surcharge', types.DoubleType(), True), 
    types.StructField('Airport_fee', types.DoubleType(), True)
])

In [ ]:
df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .parquet('yellow_tripdata_2024-10.parquet')

In [ ]:
df = df.repartition(4)


In [ ]:
df.write.mode("overwrite").parquet("yellow/2024/10/")

In [ ]:
df.printSchema()


In [ ]:
!ls -lh yellow/2024/10/*.parquet

In [ ]:
from pyspark.sql import functions as F

In [ ]:
df_15_oct = df.filter(F.to_date("tpep_pickup_datetime") == "2024-10-15")
num_trips = df_15_oct.count()
print(f"Number of taxi trips on 15th October: {num_trips}")


In [ ]:
# Calculate trip duration in hours and find the longest trip
longest_trip_hours = (
    df.withColumn(
        "trip_duration_hr",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 3600
    )
    .orderBy(F.desc("trip_duration_hr"))
    .select("trip_duration_hr")
    .limit(1)
)

longest_trip_hours.show(truncate=False)

In [ ]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv -O taxi_zone_lookup.csv


In [ ]:
# Load CSV into a DataFrame
zone_df = spark.read.csv("taxi_zone_lookup.csv", header=True, inferSchema=True)

# Show first few rows to verify
zone_df.show(5)

In [ ]:
zone_df.createOrReplaceTempView("zones_lookup")


In [ ]:
df_with_pickup_zone = df.join(zone_df, df.PULocationID == zone_df.LocationID, how="left").select(zone_df.Zone.alias("pickup_zone"))

In [ ]:

pickup_counts = (
    df_with_pickup_zone
    .groupBy("pickup_zone")
    .count()
    .orderBy("count")  # ascending order, least frequent first
)

pickup_counts.show(10)